
# Module 3: Python Programming Review: Companion Jupyter Notebook

This notebook is the hands-on companion to Module 3, the Python programming review. It
contains **every code sample from the module**, in the same order, with extra explanation
aimed at programmers who are new to Python or new to the data-science flavor of it. If
you already write Python comfortably, skim the boxed explanations and run the cells to
confirm your intuition matches the output.

**How to use this notebook:** run the cells from top to bottom, in order. Within a
section, later cells often reuse variables defined earlier in that same section, exactly
the way the original module's examples build on each other. Section boundaries generally
start fresh, so if you jump straight to the middle of a section and get a `NameError`,
scroll up to the top of that section and start there instead.

---

## Setting up your environment

You need Python 3.10 or later for one example in Section 2 (the `match`/`case`
statement). Everything else works on any reasonably recent Python 3.

Open a terminal and install the packages this notebook touches:

```bash
pip install numpy pandas matplotlib scikit-learn scipy
```

Section 9 of this notebook also uses four *developer tools* — code quality checkers, not
data science libraries. Install those too if you want to run that section's cells for
real (they will still be explained even if you skip installing them):

```bash
pip install black ruff mypy pytest
```

A quick reminder of what `pip` and `import` actually do, since this whole notebook is
built on both: `pip install <package>` downloads a package from PyPI (the Python Package
Index) onto your computer, one time. `import <package>` then loads that already-installed
package into your current Python session so you can use it — you need to `pip install`
a package exactly once per machine (or per virtual environment — more on those in Section
7), but you need to `import` it in every single script or notebook that uses it.



## Section 1 — Variables, Data Types, and Basic Operators

### Variables and dynamic typing

In Python, a variable is just a name bound to a value — you don't declare its type ahead
of time, and the same name can be rebound to a completely different type of value later.
This is called **dynamic typing**.


In [ ]:
count = 10          # count is bound to an int
print(count, type(count))

count = "ten"       # now count is bound to a str -- perfectly legal
print(count, type(count))



### Core data types

Python's handful of built-in types show up constantly: `int`, `float`, `str`, `bool`, and
the special value `None`. The built-in `type()` function tells you exactly what type a
value is.


In [ ]:
integer_value = 42
float_value = 3.14159
string_value = "hello"
boolean_value = True
nothing_value = None

print(type(integer_value))   # <class 'int'>
print(type(float_value))     # <class 'float'>
print(type(string_value))    # <class 'str'>
print(type(boolean_value))   # <class 'bool'>
print(type(nothing_value))   # <class 'NoneType'>



`None` represents the deliberate absence of a value. It's distinct from `0`, `False`, or
an empty string, even though all of those are "falsy" in a boolean context (Section 2
covers truthiness in depth). You'll see `None` constantly as a default function argument
(Section 4) and as a signal for missing data.

`isinstance()` is usually the better tool for checking a type, since it also accounts for
inheritance (which matters once you reach Section 8 on classes):


In [ ]:
print(isinstance(integer_value, int))     # True
print(isinstance(True, int))              # True -- bool is a subclass of int in Python



That last line is a classic Python quirk: booleans are technically a subtype of integers,
which is why `True + True` evaluates to `2` without raising an error. It rarely matters in
practice, but it explains behavior that would otherwise look confusing.

### Basic operators

Arithmetic works as you'd expect, with two operators worth calling out specifically: `/`
always performs **true division** and returns a float, while `//` performs **floor
division**, discarding any remainder.


In [ ]:
a, b = 17, 5

print(a + b)    # 22   addition
print(a - b)    # 12   subtraction
print(a * b)    # 85   multiplication
print(a / b)    # 3.4  true division -- always returns a float
print(a // b)   # 3    floor division -- discards the remainder
print(a % b)    # 2    modulo -- the remainder itself
print(a ** b)   # 1419857  exponentiation



A stray `/` where you meant `//` (or vice versa) is a common, quiet source of off-by-one
errors when computing array indices or batch sizes later in a data pipeline — worth
double-checking any time the result of a division feeds into an index.

String concatenation uses the same `+` operator. f-strings (covered more in Section 6)
are the standard, readable way to build formatted output — put an expression inside `{}`
directly inside a string literal prefixed with `f`.


In [ ]:
name = "Ada"
score = 97.5
print(f"{name} scored {score:.1f} points")   # Ada scored 97.5 points



### Connecting to NumPy

Python's built-in numeric types are flexible but not especially fast or memory-efficient
for large amounts of numeric data. `numpy` (installed with `pip install numpy`, almost
always imported with the community-standard alias `np`) stores numbers in tightly packed,
fixed-width memory instead, and applies the same arithmetic operators element-wise across
an entire array at once.


In [ ]:
import numpy as np

values = np.array([1, 2, 3, 4, 5], dtype=np.float64)
print(values * 2)          # [ 2.  4.  6.  8. 10.]
print(values.dtype)        # float64
print(values.dtype.itemsize)  # 8 -- bytes per element



`dtype` is worth pausing on: unlike a plain Python variable, every element in a NumPy
array shares one fixed, explicit type. Getting comfortable with the relationship between
Python's native types and NumPy's stricter, fixed-width types now will save real
confusion later, when a mismatch between an `int` and a `np.float64` produces a warning
or an unexpected result deep inside a model you're training.

### Type conversion

Python converts between types explicitly, using the type itself as a function:
`int(...)`, `float(...)`, `str(...)`.


In [ ]:
count_str = "42"
count_int = int(count_str)          # 42, as an actual integer
price_float = float("19.99")         # 19.99
back_to_str = str(count_int)          # "42"

print(count_int + 8)     # 50 -- now this works, because count_int is an int



This comes up constantly when reading data from files or user input, since everything
read from a text source arrives as a string by default, even if it visually looks like a
number. Forgetting to convert (`"42" + 8` raises a `TypeError`) is one of the most common
early errors — Python won't silently guess whether you meant string concatenation or
numeric addition, and the fix is almost always an explicit conversion like the ones above.

### Augmented assignment

Shorthand for the common pattern of updating a variable based on its own current value.


In [ ]:
total = 0
total += 5      # equivalent to: total = total + 5
total *= 2       # equivalent to: total = total * 2
print(total)     # 10



These exist for every arithmetic operator (`+=`, `-=`, `*=`, `/=`, `//=`, `**=`, `%=`), and
you'll see them constantly inside loops (Section 3), updating a running total, count, or
accumulator once per iteration.

One more distinction worth flagging now, because it resurfaces in Section 5: numbers,
strings, and booleans are all **immutable** in Python — `total += 1` doesn't modify the
integer in place, it creates a new integer and rebinds `total` to it. This is invisible
for simple values, but becomes an important, visible distinction with lists and
dictionaries, which are mutable.

### Chained comparisons

Python lets you chain comparisons in a way that reads naturally, matching ordinary
mathematical notation.


In [ ]:
age = 25
print(0 <= age < 18)    # False -- equivalent to: 0 <= age and age < 18
print(18 <= age < 65)   # True



## Section 2 — Control Flow: Conditionals

### The `if` statement

Python's conditional syntax stays close to plain English. Only one branch runs, based on
the first condition that evaluates to `True`. Notice there are no parentheses around the
condition and no braces around the block — Python uses a colon and consistent
indentation instead. Indentation isn't just style here: it's how Python knows which lines
belong to which block, so inconsistent indentation is a genuine syntax error.


In [ ]:
temperature = 72

if temperature > 80:
    print("It's hot")
elif temperature > 60:
    print("It's mild")
else:
    print("It's cold")



### Truthiness

Any value can be used as a condition, not just an explicit `True`/`False`. `0`, `0.0`,
`None`, empty strings `""`, and empty containers (`[]`, `{}`, `()`) are all **falsy**;
essentially everything else is **truthy**.


In [ ]:
data = []

if data:
    print("processing data")
else:
    print("no data to process")   # this branch runs -- an empty list is falsy



Checking a container directly (`if data:`) rather than comparing its length to zero
(`if len(data) > 0:`) is idiomatic Python, and you'll see it constantly in code you read
this week.

### Comparison and logical operators

Comparisons (`==`, `!=`, `<`, `>`, `<=`, `>=`) produce booleans, which you combine with
`and`, `or`, and `not`.


In [ ]:
age = 25
has_license = True

if age >= 18 and has_license:
    print("eligible to drive")

if not has_license:
    print("cannot drive")   # doesn't print here, since has_license is True



A subtlety worth flagging: `==` checks equality of *value*, while `is` checks *identity*
— whether two names refer to the exact same object in memory. For comparing numbers,
strings, and most everyday values, use `==`. `is` is reserved for a narrower set of cases,
most commonly checking against `None` specifically (`if result is None:`), which is the
idiomatic, recommended form.

### Ternary (conditional) expressions

A compact, single-line conditional expression, useful for simple cases.


In [ ]:
x = 7
label = "even" if x % 2 == 0 else "odd"
print(label)   # odd



This is equivalent to a full `if`/`else` block that assigns to `label` in each branch,
just condensed onto one line. It's distinctly less readable once the condition or
branches get complicated — at that point, a full `if`/`else` is the better choice.

### Membership testing with `in`

A very common pattern: checking whether a value appears inside a collection.


In [ ]:
allowed_labels = ["cat", "dog", "bird"]
label = "fish"

if label in allowed_labels:
    print("recognized label")
else:
    print("unrecognized label")   # this branch runs



`in` works on lists, tuples, strings (checking for a substring), and — as Section 5
covers — is especially fast on sets, which are built specifically for this kind of
membership check.

### Nested conditionals


In [ ]:
income = 55000
has_dependents = True

if income > 50000:
    if has_dependents:
        tax_bracket = "B-with-credit"
    else:
        tax_bracket = "B"
else:
    tax_bracket = "A"

print(tax_bracket)   # B-with-credit



When nesting gets more than two or three levels deep, it's usually a sign the logic
should be restructured — often by combining conditions with `and` directly, or by pulling
the inner logic out into its own function (Section 4). Deeply nested conditionals are a
common readability problem, precisely because each added level makes it harder to see, at
a glance, which combination of conditions leads to which outcome.

### `match`/`case`

Python 3.10+ added a structural pattern matching statement as an alternative to long
`if`/`elif` chains when comparing one value against several possibilities.


In [ ]:
status_code = 404

match status_code:
    case 200:
        print("OK")
    case 404:
        print("Not Found")
    case 500:
        print("Server Error")
    case _:
        print("Unknown status")   # the underscore is a catch-all, like else



You'll see plenty of existing code that uses long `if`/`elif` chains for this same
purpose instead — both are correct; `match`/`case` is simply a more recent, sometimes more
readable option for this specific pattern.

### Connecting to boolean masks in data science

The single most important practical extension of conditionals is **boolean indexing**,
which you'll use constantly with NumPy and pandas. Instead of writing an explicit `if` for
each element, you write a condition once, and it's evaluated across an entire array or
table at once, producing an array of booleans you can use to filter.


In [ ]:
import numpy as np
import pandas as pd

ages = np.array([15, 22, 34, 8, 61, 45])
is_adult = ages >= 18          # a boolean array, not a single True/False
print(is_adult)                # [False  True  True False  True  True]
print(ages[is_adult])          # [22 34 61 45] -- only the adult ages

df = pd.DataFrame({"age": ages, "name": ["A", "B", "C", "D", "E", "F"]})
adults_df = df[df["age"] >= 18]
print(adults_df)



There's an important gotcha the moment you want to combine more than one condition on an
array: Python's `and`/`or` keywords only work on single boolean values, not on whole
arrays. Combining boolean arrays instead requires the `&` and `|` operators, with each
individual condition wrapped in parentheses.


In [ ]:
young_adults = df[(df["age"] >= 18) & (df["age"] < 40)]
print(young_adults)



Forgetting the parentheses, or reaching for `and`/`or` out of habit, is one of the most
common early errors when filtering data with pandas — Python raises an error rather than
silently doing the wrong thing, but it's worth knowing the fix before you hit it.

### Short-circuit evaluation

`and` and `or` don't always evaluate both sides of an expression — they stop as soon as
the overall result is determined.


In [ ]:
def is_valid(x):
    print(f"checking {x}")
    return x > 0

result = is_valid(-5) and is_valid(10)   # only "checking -5" prints
print(result)   # False



`is_valid(10)` never even runs, because `is_valid(-5)` already returned `False`, and
Python knows the overall `and` can't be true regardless of the second operand. This is
routinely used deliberately as a guard: `if data and data[0] > 10:` safely checks that
`data` is non-empty before trying to access `data[0]`, and short-circuiting guarantees the
risky access never runs when `data` is empty.

### `any()` and `all()`

For checking a condition across an entire collection at once, without writing an
explicit loop.


In [ ]:
scores = [82, 91, 76, 88]

print(all(score >= 60 for score in scores))   # True -- everyone passed
print(any(score >= 90 for score in scores))   # True -- at least one A



`score >= 60 for score in scores` is a **generator expression** — a list comprehension
without the surrounding brackets, producing values one at a time rather than building a
full list in memory first, which `any()` and `all()` can stop consuming from early, the
moment they know the answer.



## Section 3 — Loops and Iteration

### The `for` loop

Python's `for` loop iterates directly over the elements of a sequence, rather than
managing an index counter by hand.


In [ ]:
fruits = ["apple", "banana", "cherry"]

for fruit in fruits:
    print(f"I like {fruit}")



When you specifically need a sequence of numbers, `range()` generates one without
building a full list in memory.


In [ ]:
for i in range(5):        # 0, 1, 2, 3, 4
    print(i ** 2)



### The `while` loop

Where a `for` loop iterates a known, fixed number of times, a `while` loop continues as
long as a condition remains true — useful when you don't know in advance how many
iterations you'll need.


In [ ]:
value = 100
steps = 0
while value > 1:
    value = value / 2
    steps += 1
print(f"took {steps} steps to shrink below 1")



Be careful with `while` loops: if the condition never becomes false, the loop runs
forever. This is a much easier mistake to make than with a `for` loop over a fixed-size
sequence, so `while` loops deserve extra scrutiny when you write one.

### `break` and `continue`

`break` exits a loop immediately; `continue` skips the rest of the current iteration and
moves on to the next one.


In [ ]:
numbers = [4, 7, 2, 9, 3, 6]

for n in numbers:
    if n > 8:
        break            # stop looping entirely once we see a number > 8
    if n % 2 != 0:
        continue          # skip odd numbers, don't print them
    print(n)



### `enumerate` and `zip`

`enumerate()` gives you both the index and the value as you iterate, avoiding a manually
maintained counter.


In [ ]:
for index, fruit in enumerate(fruits):
    print(f"{index}: {fruit}")



`zip()` lets you iterate over two (or more) sequences in lockstep.


In [ ]:
names = ["Ada", "Grace", "Alan"]
scores = [97, 88, 91]

for name, score in zip(names, scores):
    print(f"{name}: {score}")



### List comprehensions

A compact syntax for building a new list by applying an expression to every element of
an existing sequence, optionally filtering along the way.


In [ ]:
squares = [n ** 2 for n in range(10)]
even_squares = [n ** 2 for n in range(10) if n % 2 == 0]
print(squares)
print(even_squares)



This does exactly the same thing as writing an explicit `for` loop that appends to a
growing list, just more concisely — idiomatic enough that you should be comfortable
reading it even before you're fully comfortable writing it yourself.

### Dictionary and set comprehensions

The same compact syntax extends to building dictionaries and sets.


In [ ]:
names = ["Ada", "Grace", "Alan"]
name_lengths = {name: len(name) for name in names}
print(name_lengths)          # {'Ada': 3, 'Grace': 5, 'Alan': 4}

unique_first_letters = {name[0] for name in names}
print(unique_first_letters)   # {'A', 'G'}



### Nested loops

Loops can nest inside each other, which is how you iterate over every combination of two
collections, or every cell of a grid.


In [ ]:
rows = ["A", "B"]
columns = [1, 2, 3]

for row in rows:
    for column in columns:
        print(f"{row}{column}")   # A1, A2, A3, B1, B2, B3



Nested loops are a natural way to express combinatorial iteration, but they get expensive
quickly — a loop nested two levels deep over collections of size $m$ and $n$ runs its
inner block $m \times n$ times, and a third nested level multiplies that again. Whenever
you find yourself writing a doubly or triply nested loop over numeric data specifically,
it's worth checking whether a NumPy operation could replace it.

### A brief note on `itertools`

For less common iteration patterns, the standard library's `itertools` module (built in
to Python — no `pip install` needed) provides efficient, well-tested tools rather than
requiring you to hand-write nested loops for the same purpose.


In [ ]:
from itertools import product

for row, column in product(["A", "B"], [1, 2, 3]):
    print(f"{row}{column}")   # same output as the nested loop above



### Why explicit loops give way to vectorized operations

Here's the practical payoff for this course. Suppose you want to square every element of
a large numeric collection, written as an explicit Python loop.


In [ ]:
import time

numbers_list = list(range(1_000_000))

start = time.time()
squared = [n ** 2 for n in numbers_list]
print(f"pure Python: {time.time() - start:.4f} seconds")



Compare that to the equivalent NumPy operation.


In [ ]:
import numpy as np

numbers_array = np.arange(1_000_000)

start = time.time()
squared_np = numbers_array ** 2
print(f"NumPy vectorized: {time.time() - start:.4f} seconds")



Run both, and you'll typically see the NumPy version finish tens of times faster (exact
numbers vary by machine). The reason connects directly back to Section 1: a Python `for`
loop processes one Python object at a time, with all the overhead that implies, while
NumPy's vectorized operations push the entire computation down into fast, compiled,
fixed-width-typed code, operating on the whole array at once. Throughout the labs,
whenever you catch yourself writing an explicit Python loop over a NumPy array or a
pandas column, it's worth pausing to ask whether a vectorized operation could do the same
thing — often it can, and it will usually be both shorter to write and dramatically
faster to run.

### A specific pandas pitfall worth knowing now

pandas DataFrames support `.iterrows()`, which lets you loop over rows one at a time,
much like looping over a plain list.


In [ ]:
import pandas as pd

df = pd.DataFrame({"x": [1, 2, 3, 4]})

for index, row in df.iterrows():
    print(index, row["x"])



This works, but it's usually the slowest possible way to process a DataFrame's contents,
for exactly the reason described above. Nearly every task that looks like it needs
`.iterrows()` — computing a new column from existing ones, filtering rows, aggregating
values — has a vectorized equivalent (`df["x"] * 2`, boolean masking from Section 2,
`.apply()` from Section 4, or a built-in aggregation method) that will run faster and is
usually shorter to write. Treat `.iterrows()` as a last resort, not a default habit.



## Section 4 — Functions

Functions are how you package a block of code into a reusable, named unit. Nearly every
tool you've used already in this course — `LinearRegression().fit()`,
`mean_squared_error()`, `np.cov()` — is a function (or a method, a function attached to an
object, covered in Section 8).

### Defining a function

`def` introduces a function, followed by a name, a parenthesized list of parameters, and
an indented block. `return` sends a value back to the caller and immediately exits the
function. A function with no explicit `return` returns `None` by default — a common
source of confusion when you forget to return a value you meant to compute.


In [ ]:
def greet(name):
    return f"Hello, {name}!"

message = greet("Ada")
print(message)   # Hello, Ada!



### Parameters and default arguments

Parameters can have default values, making them optional at the call site.


In [ ]:
def scale_value(x, factor=1.0, offset=0.0):
    return x * factor + offset

print(scale_value(10))                 # 10.0 -- uses both defaults
print(scale_value(10, factor=2.0))     # 20.0
print(scale_value(10, factor=2.0, offset=5.0))  # 25.0



Calling with `factor=2.0` explicitly (a **keyword argument**) rather than just `2.0` (a
**positional argument**) is optional here, but becomes important readability practice once
a function has several parameters — you'll see this convention everywhere in
scikit-learn's API, where nearly every argument beyond the first is expected to be passed
by keyword.

### `*args` and `**kwargs`

Occasionally you want a function to accept an arbitrary number of positional or keyword
arguments, without naming each one in advance.


In [ ]:
def summarize(*values, **labels):
    print(f"values: {values}")   # a tuple of all positional arguments
    print(f"labels: {labels}")   # a dict of all keyword arguments

summarize(1, 2, 3, unit="meters", precision=2)



You won't write `*args`/`**kwargs` constantly, but you'll see it often in library code —
it's how many scikit-learn and pandas functions forward arguments through to lower-level
routines — so recognizing the syntax matters even if you rarely produce it yourself.

### Docstrings

A string literal placed immediately inside a function (before any other code) becomes its
**docstring** — documentation that tools, IDEs, and the built-in `help()` function can
retrieve automatically.


In [ ]:
def z_score(x, mean, std):
    '''Standardize a value using the given mean and standard deviation.

    Returns (x - mean) / std.
    '''
    return (x - mean) / std

help(z_score)



Get in the habit of writing at least a one-line docstring for any function whose purpose
isn't immediately obvious from its name.

### Scope

A variable created inside a function is **local** to that function — it doesn't exist
outside it, and it doesn't overwrite a same-named variable defined elsewhere.


In [ ]:
total = 100

def reset_total():
    total = 0        # this creates a new, local "total" -- it does NOT change the outer one
    return total

print(reset_total())   # 0
print(total)            # 100 -- unchanged



This is a deliberate safety feature: a function's internal variables can't accidentally
clobber variables in the code that calls it.

### The `global` keyword

Occasionally you genuinely want a function to modify a variable defined outside it,
rather than creating a new local one. Python requires you to say so explicitly.


In [ ]:
total = 100

def add_to_total(amount):
    global total
    total += amount

add_to_total(50)
print(total)   # 150 -- the outer variable really was changed this time



In practice, relying on `global` is usually a sign a function should instead take the
value it needs as a parameter and return the updated result, rather than reaching outside
itself to modify shared state — code that only reads and writes its own parameters and
local variables is far easier to reason about and test in isolation.

### Type hints

Python remains dynamically typed, but you can optionally annotate a function's expected
parameter and return types as documentation, which tools and IDEs can check for you
without changing how the code actually runs (Section 9 shows a tool, `mypy`, that
actually checks these).


In [ ]:
def z_score(x: float, mean: float, std: float) -> float:
    return (x - mean) / std

print(z_score(10.0, 5.0, 2.0))



Python doesn't enforce these annotations at runtime by default — passing a string where
`float` is annotated won't raise an error on its own — but they make a function's
expected inputs and outputs self-documenting.

### Recursive functions

A function can call itself, a technique called **recursion**, most naturally suited to
problems that break down into smaller versions of themselves.


In [ ]:
def factorial(n):
    if n <= 1:
        return 1                     # base case -- stops the recursion
    return n * factorial(n - 1)      # recursive case -- calls itself

print(factorial(5))   # 120



Every recursive function needs a **base case** — a condition where it returns a value
directly, without calling itself again — or it will recurse forever (and eventually crash
with a `RecursionError`).

### A common gotcha: mutable default arguments

This one is worth flagging directly, because it produces genuinely surprising behavior
the first time you hit it. Default argument values are evaluated exactly once, when the
function is *defined* — not once per call — which causes trouble if that default is a
mutable object like a list.


In [ ]:
def add_item(item, collection=[]):     # BUG: the same list is reused every call
    collection.append(item)
    return collection

print(add_item("a"))   # ['a']
print(add_item("b"))   # ['a', 'b'] -- surprising! it's the SAME list as before



The standard fix is to use `None` as the default and create a new, empty collection
inside the function body when needed.


In [ ]:
def add_item_fixed(item, collection=None):
    if collection is None:
        collection = []
    collection.append(item)
    return collection

print(add_item_fixed("a"))   # ['a']
print(add_item_fixed("b"))   # ['b'] -- a fresh list each time, as expected



This is exactly the idiomatic `is None` check from Section 2, and it's one of the most
commonly cited "gotchas" in Python for good reason — the bug is easy to introduce and,
because it only shows up after multiple calls, easy to miss during quick testing.

### Lambda functions

For short, throwaway functions, Python offers an inline syntax that skips the `def`
keyword and the function name entirely.


In [ ]:
square = lambda x: x ** 2
print(square(5))   # 25



### Functions as first-class values

In Python, a function is a value like any other: you can pass it as an argument to
another function, store it in a variable, or put it in a list. This is the mechanism
behind one of the most common data science patterns you'll use this week — applying a
function across every row or column of a pandas DataFrame.


In [ ]:
import pandas as pd

df = pd.DataFrame({"celsius": [0, 20, 37, 100]})

def to_fahrenheit(c):
    return c * 9 / 5 + 32

df["fahrenheit"] = df["celsius"].apply(to_fahrenheit)
print(df)



`.apply()` takes your function itself — not the result of calling it — and calls it once
per element, exactly the way a `for` loop from Section 3 would, but far more concisely.
The same pattern, using a lambda instead of a separately defined function, is common for
very short transformations.


In [ ]:
df["fahrenheit_lambda"] = df["celsius"].apply(lambda c: c * 9 / 5 + 32)
print(df)



Recognizing that `.apply()`, `.map()`, and similar pandas/NumPy methods are just "call
this function, once per element" — the same idea as a loop, handed off to a library that
can execute it faster — will make a large fraction of the data-processing code you
encounter this week feel far less mysterious.



## Section 5 — Core Data Structures: Lists, Tuples, Dictionaries, and Sets

### Lists

A list is an ordered, mutable (changeable) collection, written with square brackets.


In [ ]:
scores = [85, 92, 78, 90]

scores.append(88)          # add an element to the end
scores[0] = 87              # change an existing element by index
print(scores[-1])           # 88 -- negative indices count from the end
print(scores[1:3])          # [92, 78] -- slicing: elements at index 1 and 2
print(scores)



Lists are the default, general-purpose collection in Python, used constantly to
accumulate results, hold sequences of mixed or uniform data, and pass collections of
values around your programs.

### Tuples

A tuple looks similar to a list but is **immutable** — once created, its contents can't
be changed — and is written with parentheses (or often no brackets at all).


In [ ]:
point = (3, 4)
x, y = point                # unpacking: assigns 3 to x and 4 to y in one line
print(x, y)                 # 3 4



Immutability is a feature, not a limitation: it signals that this value is meant to stay
fixed, and it makes tuples usable as dictionary keys (which lists, being mutable, cannot
be). Tuples are also common as a function's return value, when it needs to hand back more
than one result at once.


In [ ]:
def min_and_max(values):
    return min(values), max(values)   # returns a tuple

lowest, highest = min_and_max([4, 8, 1, 9, 3])
print(lowest, highest)    # 1 9



### Dictionaries

A dictionary stores key-value pairs, giving you fast lookup by key rather than by
numeric position.


In [ ]:
student = {"name": "Ada", "age": 28, "major": "Computer Science"}

print(student["name"])        # Ada
student["age"] = 29            # update an existing key
student["gpa"] = 3.9            # add a new key

for key, value in student.items():
    print(f"{key}: {value}")



Dictionaries are the natural way to represent a single structured record — one row of
data with named fields — and a list of dictionaries is one of the most common ways
real-world data arrives before it's loaded into a more specialized structure like a
pandas DataFrame, as you'll see below.

### Sets

A set is an unordered collection of unique elements — useful whenever you care about
membership or uniqueness rather than order or position.


In [ ]:
tags_a = {"python", "data", "ml"}
tags_b = {"ml", "stats", "python"}

print(tags_a | tags_b)     # union: {'python', 'data', 'ml', 'stats'}
print(tags_a & tags_b)     # intersection: {'python', 'ml'}
print(tags_a - tags_b)     # difference: {'data'}

print("python" in tags_a)  # True -- membership testing is very fast on a set



### Choosing the right structure

A quick way to decide: use a **list** when order matters and you might change the
contents; a **tuple** when the values are fixed and won't change; a **dictionary** when
you need to look values up by a meaningful name rather than a position; and a **set** when
you care about uniqueness or fast membership testing and don't care about order.

### Sorting

`sorted()` works on any of these collections and accepts a `key` argument — a function,
tying directly back to Section 4's point that functions are ordinary values — telling it
what to sort by.


In [ ]:
students = [
    {"name": "Ada", "score": 91},
    {"name": "Grace", "score": 88},
    {"name": "Alan", "score": 95},
]

by_score = sorted(students, key=lambda s: s["score"], reverse=True)
for s in by_score:
    print(s["name"], s["score"])
# Alan 95
# Ada 91
# Grace 88



`sorted()` returns a new, sorted list without changing the original; lists also have an
in-place `.sort()` method that modifies the list directly and returns `None` — a common
small mistake is writing `students = students.sort()`, which sets `students` to `None`,
since `.sort()` doesn't return the sorted list the way `sorted()` does.

### Nested structures

These four collection types combine freely: a list of dictionaries, a dictionary of
lists, a dictionary whose values are themselves dictionaries, and so on. Real-world
structured data, especially data loaded from JSON, routinely nests two or three levels
deep.


In [ ]:
course = {
    "name": "Data Science Intensive",
    "students": [
        {"name": "Ada", "scores": [91, 88, 95]},
        {"name": "Grace", "scores": [88, 90, 85]},
    ],
}

for student in course["students"]:
    average = sum(student["scores"]) / len(student["scores"])
    print(f"{student['name']}: {average:.1f}")



Reading nested structures like this is mostly a matter of working outward to inward, one
level of `[]` or `.key` access at a time — exactly the pattern in the loop above, which
reaches into `course["students"]`, then into each `student["scores"]`.

### A note on copying mutable structures

Because lists and dictionaries are mutable, assigning one to a new name doesn't create an
independent copy — both names refer to the exact same underlying object.


In [ ]:
original = [1, 2, 3]
alias = original          # alias is NOT a copy -- it's the same list
alias.append(4)
print(original)            # [1, 2, 3, 4] -- the "original" changed too!



To get an actual independent copy, use `.copy()` (or the `list()`/`dict()` constructors),
keeping in mind that this copies only the outer structure — a **shallow copy**. If the
structure contains nested mutable objects (a list of lists, for instance), those inner
objects are still shared between the original and the copy unless you use
`copy.deepcopy()` instead. This subtlety trips up experienced programmers just as often as
beginners — worth testing directly, as above, whenever you're not sure whether an
operation is going to modify data you didn't mean to touch.

### Connecting to data science structures

A list of dictionaries — each dictionary representing one record with the same set of
keys — converts directly into a pandas DataFrame.


In [ ]:
import pandas as pd

records = [
    {"name": "Ada", "age": 28, "score": 91},
    {"name": "Grace", "age": 34, "score": 88},
    {"name": "Alan", "age": 41, "score": 95},
]

df = pd.DataFrame(records)
print(df)



And a plain Python list of numbers converts directly into a NumPy array, which — as
Section 1 covered — packs the same numbers into a more compact, fixed-type, much faster
structure.


In [ ]:
import numpy as np

plain_list = [4, 8, 15, 16, 23, 42]
array_version = np.array(plain_list)
print(array_version.mean())   # 18.0



Dictionaries also show up constantly as a way to hold configuration or hyperparameters
for a model before passing them along.


In [ ]:
from sklearn.linear_model import Ridge

hyperparameters = {"alpha": 1.0, "fit_intercept": True, "random_state": 0}
model = Ridge(**hyperparameters)   # unpack the dict directly into keyword arguments
print(model)



That `**hyperparameters` syntax is the dictionary counterpart to the `**kwargs` you saw
in Section 4 — it unpacks a dictionary's key-value pairs directly into a function call's
keyword arguments, which is exactly how the line above hands `alpha=1.0,
fit_intercept=True, random_state=0` to `Ridge` in a single, easily-modified line.



## Section 6 — Strings, File I/O, and Error Handling

### String basics

Strings support indexing and slicing just like lists, along with a large set of built-in
methods for common transformations.


In [ ]:
text = "Data Science"

print(text.lower())          # data science
print(text.upper())          # DATA SCIENCE
print(text.split())          # ['Data', 'Science']
print(text.replace("Data", "Computer"))   # Computer Science
print(text[0:4])             # Data -- slicing works the same as with lists
print(len(text))             # 12



A few more string methods come up often enough to be worth knowing now: `.strip()`
removes leading and trailing whitespace (extremely common when cleaning text read from a
file, which often carries stray newline characters), and `.join()` does the reverse of
`.split()`, combining a list of strings into one, with a specified separator in between.


In [ ]:
messy = "   raw input line \n"
clean = messy.strip()
print(repr(clean))     # 'raw input line'  -- note: repr() shows quotes and escapes

words = ["Data", "Science", "Intensive"]
joined = "-".join(words)
print(joined)           # Data-Science-Intensive



### f-strings

f-strings are the standard, modern way to build formatted strings, embedding expressions
directly inside `{}` placeholders.


In [ ]:
name = "model_v2"
accuracy = 0.9137

print(f"{name} achieved {accuracy:.2%} accuracy")   # model_v2 achieved 91.37% accuracy



The `:.2%` inside the placeholder is a **format specifier** — here, "format this number as
a percentage with two decimal places." Format specifiers worth knowing for numeric output
specifically: `.2f` for two decimal places, `,` for thousands separators, and `%` for
percentages are the ones you'll reach for constantly when reporting model metrics.

### Reading and writing files

Python's built-in `open()` function, used with a `with` block, is the standard way to
work with files.


In [ ]:
with open("notes.txt", "w") as f:
    f.write("first line\n")
    f.write("second line\n")

with open("notes.txt", "r") as f:
    contents = f.read()
print(contents)



The `with` statement is a **context manager**: it guarantees the file is properly closed
once the block finishes, even if an error occurs partway through — you don't need to
remember to call `f.close()` yourself. You'll see this same `with` pattern used for other
resources later in the course (database connections, for instance) that need reliable
cleanup.

### Reading large files line by line

`f.read()` loads an entire file into memory as one string, fine for small files but
wasteful — or outright impossible — for files too large to fit comfortably in memory.
Iterating over the file object directly instead reads it one line at a time.


In [ ]:
with open("notes.txt", "r") as f:
    for line in f:
        print(line.strip())    # .strip() removes the trailing newline character



This is the same for-over-an-iterable pattern from Section 3, applied to a file instead
of a list — the file object itself is iterable, yielding one line per iteration, without
ever holding the whole file in memory at once.

### File paths with `pathlib`

Rather than building file paths as plain strings, the standard library's `pathlib`
module (built in, no install needed) represents paths as objects with their own
convenient methods.


In [ ]:
from pathlib import Path

data_dir = Path(".")
file_path = data_dir / "scores.csv"    # the / operator joins path components

print(file_path)              # scores.csv
print(file_path.exists())      # True or False, depending on whether it's there
print(file_path.suffix)        # .csv



`pathlib` handles the differences between operating systems' path separators for you (`/`
on macOS and Linux, `\` on Windows) and is the modern, recommended way to build and check
file paths, rather than manually concatenating strings with `+`.

### Reading data the manual way, versus with pandas

It's worth seeing what `pandas.read_csv()` is actually saving you from doing by hand.
First, create a small CSV file to work with.


In [ ]:
with open("scores.csv", "w") as f:
    f.write("name,age,score\n")
    f.write("Ada,28,91\n")
    f.write("Grace,34,88\n")
    f.write("Alan,41,95\n")



A very simple, manual CSV reader:


In [ ]:
with open("scores.csv", "r") as f:
    lines = f.read().strip().split("\n")

header = lines[0].split(",")
rows = [line.split(",") for line in lines[1:]]
print(header)
print(rows)



This works for the simplest possible CSV files, but it breaks quickly: values inside
quotes that contain commas, missing fields, inconsistent whitespace, and mixed data types
(should `"92"` become the string `"92"` or the integer `92`?) all require handling that a
hand-rolled reader like the one above doesn't attempt. This is exactly what
`pandas.read_csv()` handles for you.


In [ ]:
import pandas as pd

df = pd.read_csv("scores.csv")
print(df.head())
print(df.dtypes)     # pandas has already inferred a sensible type per column



### Error handling

Things go wrong — a file might not exist, a value might not convert to the number you
expected, a list index might be out of range. Python signals these problems by raising
**exceptions**, and you handle them with `try`/`except`.


In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("cannot divide by zero, returning None instead")
        return None

print(safe_divide(10, 2))   # 5.0
print(safe_divide(10, 0))   # prints a message, returns None



Catching a specific exception type (`ZeroDivisionError` above) rather than every possible
error is the recommended practice — a bare `except:` silently swallows errors you didn't
anticipate, including genuine bugs, which makes them much harder to track down later.

### Multiple `except` clauses and `finally`

A single `try` block can handle several different kinds of errors, each with its own
`except` clause, and an optional `finally` block that runs no matter what.


In [ ]:
def load_and_parse(path):
    try:
        with open(path, "r") as f:
            contents = f.read()
        return int(contents.strip())
    except FileNotFoundError:
        print(f"file not found: {path}")
        return None
    except ValueError:
        print(f"file did not contain a valid integer")
        return None
    finally:
        print("finished attempting to load the file")   # always runs

# demonstrate all three paths: missing file, bad content, and success
load_and_parse("does_not_exist.txt")

with open("bad_number.txt", "w") as f:
    f.write("not a number")
load_and_parse("bad_number.txt")

with open("good_number.txt", "w") as f:
    f.write("42")
print(load_and_parse("good_number.txt"))



Ordering matters here: Python checks `except` clauses in order and uses the first one
that matches the exception that was actually raised, so put more specific exception types
before more general ones if you have both.

### Raising your own exceptions

You can also raise your own exceptions, to signal that a function received input it
can't sensibly handle.


In [ ]:
def load_scores(values):
    if not values:
        raise ValueError("values list cannot be empty")
    return sum(values) / len(values)

try:
    load_scores([])
except ValueError as e:
    print(f"caught an error: {e}")



### Custom exception classes

For larger programs, it's common to define your own exception types — a preview of the
class syntax Section 8 covers in depth — so callers can distinguish your specific error
conditions from Python's generic built-in ones.


In [ ]:
class InsufficientDataError(Exception):
    '''Raised when there isn't enough data to compute a reliable result.'''
    pass

def compute_average(values, minimum_count=5):
    if len(values) < minimum_count:
        raise InsufficientDataError(
            f"need at least {minimum_count} values, got {len(values)}"
        )
    return sum(values) / len(values)

try:
    compute_average([1, 2])
except InsufficientDataError as e:
    print(f"caught a custom error: {e}")



You'll use exactly this pattern constantly in the labs when loading external data: wrap
the loading step in a `try`/`except`, and fail loudly and specifically (rather than
silently producing wrong results) when a file is missing, malformed, or otherwise not
what your code expects.



## Section 7 — Modules, Packages, and the Python Data Science Stack

### Modules and the `import` statement

A **module** is simply a Python file containing definitions — functions, variables,
classes — that other files can reuse. `import` makes a module's contents available in
your current file. `math` is part of Python's standard library, so it needs no `pip
install` step.


In [ ]:
import math

print(math.sqrt(16))     # 4.0
print(math.pi)            # 3.141592653589793



You can import a specific name directly out of a module, avoiding the need to prefix it
every time.


In [ ]:
from math import sqrt, pi

print(sqrt(25))    # 5.0



And you can give an imported module a shorter alias — a convention you've seen throughout
this entire course already.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("all three imported with their standard aliases")



These specific aliases (`np`, `pd`, `plt`) aren't required by Python itself — they're a
near-universal community convention, strong enough that seeing `np.array(...)` or
`pd.DataFrame(...)` in someone else's code tells you immediately, without checking the
imports, exactly which library `np` and `pd` refer to. Following this convention, rather
than inventing your own aliases, makes your code dramatically easier for anyone else
(including a future version of you) to read.

### Packages and PyPI

A **package** is a collection of modules bundled together and distributed as a single
installable unit. The **Python Package Index (PyPI)** is the central public repository
where most open-source Python packages, including every library used in this course, are
published. `pip`, Python's package installer, downloads and installs packages from PyPI.
This is a terminal command, not Python code — run it in your shell, not in a notebook
cell:

```bash
pip install numpy pandas matplotlib scikit-learn scipy
```

Once installed, a package's modules are available to `import` in any Python program on
your system (or, more precisely, within whichever **virtual environment** you installed
it into).

### Why virtual environments matter

Different projects often need different, sometimes conflicting, versions of the same
package. A virtual environment is an isolated, self-contained installation of Python and
its packages, so that installing or upgrading a package for one project never affects any
other project on the same machine. Again, these are terminal commands, run once when
setting up a project — not something you execute inside this notebook:

```bash
python -m venv course_env             # create a new virtual environment
source course_env/bin/activate         # activate it (macOS/Linux)
pip install numpy pandas scikit-learn   # installs only into this environment
```

Once you've settled on the packages a project needs, `pip freeze` records their exact
versions into a `requirements.txt` file, which lets anyone else (or you, on a different
machine) recreate the identical environment:

```bash
pip freeze > requirements.txt
pip install -r requirements.txt        # recreates the same environment elsewhere
```

### Version numbers

Most Python packages, including every one in this course's stack, follow **semantic
versioning** — a version number like `2.1.4` where the first number changes for breaking,
incompatible changes, the second for new features that remain backward-compatible, and
the third for bug fixes. Pinning a specific version in `requirements.txt`
(`pandas==2.1.4` rather than just `pandas`) is common practice for any code you want to
keep running identically over time, since a library's newer major version can
occasionally change behavior in ways that break existing code.

### The standard library versus PyPI

Python ships with a large **standard library** — modules like `math`, `itertools`
(Section 3), and `os` — available with no installation step at all, simply because Python
itself came with them. Everything from this course's core data science stack (NumPy,
pandas, Matplotlib, SciPy, scikit-learn), by contrast, is a third-party package that must
be installed separately via `pip`, precisely because Python's standard library is
deliberately kept general-purpose rather than specialized for any one domain like data
science.

### The `if __name__ == "__main__":` idiom

Every Python file has a built-in `__name__` variable, automatically set to `"__main__"`
if that file is the one you ran directly, and set to the module's actual name if that
file was instead `import`-ed by some other file.


In [ ]:
def main():
    print("running the main program")

if __name__ == "__main__":
    main()

print(f"in this notebook, __name__ is: {__name__!r}")



This idiom lets a file be used two ways at once — run directly as a standalone script,
or imported elsewhere purely for its function and class definitions, without accidentally
re-running its main logic just because it was imported. You won't need this constantly
for short lab notebooks (as you can see above, `__name__` is already `'__main__'` inside a
running notebook), but you'll see it in nearly every substantial `.py` script you read
this week.

### `pip` versus `conda`

`pip` is Python's own package installer and pulls from PyPI, as described above. In data
science specifically, you'll also frequently encounter `conda`, a separate package and
environment manager that (unlike `pip`) can install non-Python dependencies too — this
matters because some scientific libraries rely on lower-level, compiled components
(BLAS/LAPACK linear algebra libraries, for instance, which NumPy and SciPy build on) that
conda can manage directly, while pip-installed packages typically bundle or assume those
are already present. Many data science environments use conda specifically for this
reason, though pip remains the more universal choice across the wider Python ecosystem.

### Checking what's installed

`pip show <package>` (a terminal command) reports a package's installed version and where
it lives. Checking a library's `__version__` attribute directly from within Python
confirms exactly what's loaded in your current session — often the first thing worth
checking when code that "should work" doesn't.


In [ ]:
import numpy as np
import sklearn

print(np.__version__)
print(sklearn.__version__)



### A quick tour of this course's core stack

You've already used every one of these libraries; here's a minimal example touching all
five at once, showing how naturally they chain together.


In [ ]:
import numpy as np              # arrays and fast numerical computation
import pandas as pd             # labeled tabular data (DataFrames)
import matplotlib.pyplot as plt # plotting and visualization
import scipy                    # additional scientific/statistical routines
import sklearn                  # machine learning models and evaluation tools

# a minimal example touching all five
rng = np.random.default_rng(0)
data = pd.DataFrame({"x": np.linspace(0, 10, 50)})
data["y"] = 2 * data["x"] + rng.normal(0, 1, 50)

from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(data[["x"]], data["y"])

plt.scatter(data["x"], data["y"])
plt.plot(data["x"], model.predict(data[["x"]]), color="red")
plt.show()



Notice how naturally these libraries chain together in that example: a pandas DataFrame
holds the data, NumPy generates the synthetic values and noise, scikit-learn fits the
model, and Matplotlib displays the result — each library handling the piece it's
specifically built for, all interoperating through the shared NumPy array underneath.
This interoperability, more than any single feature of any one library, is why this
specific combination has become the default toolkit for data science in Python.



## Section 8 — Object-Oriented Programming

Object-oriented programming (OOP) bundles data and the functions that operate on it
together into a single unit, called an **object**. You've been using objects constantly
throughout this course already, every time you called `.fit()` on a scikit-learn model or
`.mean()` on a NumPy array. This section shows you how to build your own.

### Classes and instances

A **class** is a blueprint for creating objects; an **instance** is one specific object
built from that blueprint.


In [ ]:
class Dog:
    def __init__(self, name, breed):
        self.name = name      # an attribute -- data belonging to this instance
        self.breed = breed

    def bark(self):
        return f"{self.name} says Woof!"

my_dog = Dog("Rex", "Labrador")     # creates an instance
print(my_dog.name)                   # Rex
print(my_dog.bark())                 # Rex says Woof!



`__init__` is a special method — called a **constructor** — that runs automatically when
a new instance is created, responsible for setting up that instance's initial attributes.
`self` refers to the specific instance the method is being called on; it's always the
first parameter of a method, and Python passes it automatically — you never supply it
yourself when calling `my_dog.bark()`.

### Instance attributes versus class attributes

Everything assigned to `self` inside `__init__` is an **instance attribute** — its own
separate copy exists for every instance. A **class attribute**, defined directly inside
the class body rather than inside a method, is instead shared by every instance of the
class.


In [ ]:
class Dog:
    species = "Canis familiaris"     # class attribute -- shared by all dogs

    def __init__(self, name, breed):
        self.name = name              # instance attribute -- unique per dog
        self.breed = breed

rex = Dog("Rex", "Labrador")
fido = Dog("Fido", "Poodle")

print(rex.species, fido.species)   # Canis familiaris Canis familiaris -- shared
print(rex.name, fido.name)          # Rex Fido -- distinct per instance



Class attributes are the right choice for a value that's genuinely the same across every
instance (a constant, a default configuration, a shared counter); anything specific to
one particular object belongs in `__init__` as an instance attribute instead.

A note on encapsulation: Python doesn't enforce truly private attributes the way some
languages do, but it has a strong naming convention — a single leading underscore
(`self._internal_value`) signals "this is an implementation detail, please don't access
it directly from outside the class," even though nothing technically prevents you from
doing so. You'll see this same convention used throughout scikit-learn's source code.

### Inheritance

A class can be built as a specialized version of another class, inheriting its
attributes and methods and adding or overriding specific pieces.


In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

class Cat(Animal):                   # Cat inherits from Animal
    def speak(self):                  # override: cats speak differently
        return f"{self.name} says Meow!"

generic_animal = Animal("Creature")
my_cat = Cat("Whiskers")

print(generic_animal.speak())   # Creature makes a sound
print(my_cat.speak())            # Whiskers says Meow!



This pattern — a shared base class defining a common interface, with specific subclasses
providing their own implementation of a method — is called **polymorphism**, and it's
precisely the design underlying scikit-learn's consistency: `LinearRegression`, `Ridge`,
and every other scikit-learn model share the same base interface (`fit()`, `predict()`),
while each one implements what those methods actually do internally very differently.

### Dunder methods

Methods surrounded by double underscores, like `__init__`, are called **dunder methods**
(short for "double underscore"), and Python calls them automatically in specific
situations rather than you calling them directly.


In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"    # controls how print() displays it

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y   # controls ==

p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1)             # Point(1, 2) -- uses __repr__, not a generic object address
print(p1 == p2)        # True -- uses __eq__, comparing values, not object identity



Without `__repr__`, printing an instance would show something unhelpful like
`<__main__.Point object at 0x7f...>`; without `__eq__`, `==` would fall back to identity
comparison (are these literally the same object in memory?), so two separately created
points with identical coordinates would compare as unequal. Defining these two dunder
methods is common practice for almost any class you write that represents a simple value.

### Building your own scikit-learn-style estimator

To make that connection completely concrete, here's a small class that mimics the
`fit()`/`predict()` interface you've used throughout this course, implementing ordinary
least squares by hand using exactly the normal-equations formula from the statistics
module.


In [ ]:
import numpy as np

class SimpleLinearModel:
    def __init__(self):
        self.beta_ = None      # trailing underscore: a scikit-learn convention
                                 # for attributes only set after fitting

    def fit(self, X, y):
        X_design = np.column_stack([np.ones(len(X)), X])
        self.beta_ = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y
        return self             # returning self allows method chaining

    def predict(self, X):
        X_design = np.column_stack([np.ones(len(X)), X])
        return X_design @ self.beta_

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 50)
y = 2.5 * x + 3.0 + rng.normal(0, 2.0, 50)

model = SimpleLinearModel()
model.fit(x, y)
predictions = model.predict(x)
print(f"fitted parameters: {model.beta_}")



Every piece of this should look familiar: `__init__` sets up the object's initial state,
`fit()` stores what it learns as attributes on `self` (using the trailing-underscore
naming convention scikit-learn itself follows for exactly this purpose), and `predict()`
uses those stored attributes to generate new output. Writing this class by hand should
make what happens inside `LinearRegression().fit(X, y)` feel considerably less like a
black box — it's the same normal-equations computation, simply packaged as a reusable
object with a standardized interface.



## Section 9 — Techniques for Improving Code Quality

Everything in Sections 1 through 8 was about getting code to produce a correct result.
This section is about making that code reliable, readable, and safe to change later — by
you, six months from now, or by someone else on your team next week.

The four tools below (`black`, `ruff`, `mypy`, `pytest`) are installed the same way as
any other package:

```bash
pip install black ruff mypy pytest
```

The cells below actually run these tools against small example files, so you can see the
real output rather than just a transcript. If you haven't installed the tools, the
explanatory text still tells you what each one is for.

### Style consistency and formatters

Python has an official style guide, PEP 8, covering things like spacing, naming
conventions, and line length. Rather than memorizing and manually applying it, the
standard practice is to run an automatic code formatter — most commonly `black` — which
rewrites your code into a consistent style without changing what it does. We write a
messy file to disk, then let `black` clean it up.

(The `%%writefile` line below is a Jupyter "magic command," not Python code — it tells
the notebook to save everything else in the cell to a file instead of running it as
Python. The `!` prefix on other lines runs a shell command directly from the notebook,
exactly as if you'd typed it into a terminal.)


In [ ]:
%%writefile messy_example.py
def compute_average(values):
 total=0
 for i in range(len(values)):
     total = total+values[i]
 average=total/len(values)
 return average


In [ ]:
!python3 -m black messy_example.py


In [ ]:
with open("messy_example.py") as f:
    print(f.read())



The value of an automatic formatter isn't aesthetic — it's that it removes style entirely
as a topic of discussion. Nobody argues about spacing in a code review once a formatter
enforces it automatically on every save or commit.

### Linters: catching mistakes before you ever run the code

A **linter** performs static analysis — reading your code without executing it — to flag
likely bugs, unused code, and style violations. `ruff` is a fast, widely used one.


In [ ]:
%%writefile bad_example.py
import numpy as np
import pandas as pd

def compute_average(values):
    total = 0
    for i in range(len(values)):
        total = total + values[i]
    average = total / len(values)
    return average

result = compute_average([1, 2, 3, 4, 5])
print(result)


In [ ]:
!python3 -m ruff check bad_example.py



Neither unused import (`numpy`, `pandas`) would cause an error when you actually ran this
script — Python doesn't care if you import something you never use. That's precisely the
point of a linter: it catches things that are technically valid but almost certainly
mistakes (a forgotten import you meant to delete, a variable you assigned but never read,
a comparison that's always true), before they accumulate into genuine confusion about
what a file actually depends on.

### Type checkers

Section 4 introduced type hints (`def z_score(x: float, mean: float, std: float) ->
float:`) as optional documentation that Python doesn't enforce at runtime. A type checker
like `mypy` reads those same hints and checks, statically, whether your code is actually
consistent with them — catching a whole class of bugs before you ever run the program.


In [ ]:
%%writefile typed_example.py
def z_score(x: float, mean: float, std: float) -> float:
    return (x - mean) / std

result = z_score(10, 5, "2")   # passing a string where a float is expected


In [ ]:
!python3 -m mypy typed_example.py



This error is caught by reading the code, not by running it — which means it surfaces
immediately, in your editor, rather than showing up later as a confusing runtime failure
deep inside a data pipeline.

### Automated (unit) testing

Linters and type checkers catch mistakes about form; they can't tell you whether your
function computes the right answer. That's the job of a **test**: a small piece of code
that calls your function with known inputs and checks the result against a known correct
output. `pytest` is the standard tool for this in Python.


In [ ]:
%%writefile stats_utils.py
def z_score(x, mean, std):
    if std == 0:
        raise ValueError("standard deviation cannot be zero")
    return (x - mean) / std


In [ ]:
%%writefile test_stats_utils.py
from stats_utils import z_score
import pytest

def test_z_score_basic():
    assert z_score(10, 5, 2) == 2.5

def test_z_score_zero_mean():
    assert z_score(0, 0, 1) == 0.0

def test_z_score_raises_on_zero_std():
    with pytest.raises(ValueError):
        z_score(10, 5, 0)


In [ ]:
!python3 -m pytest test_stats_utils.py -v



Notice the third test: it deliberately checks that calling the function with bad input
(`std=0`) raises the exception you expect, using `pytest.raises` — a good test suite
verifies failure behavior, not just success cases. The real payoff of a test suite like
this arrives the second time you touch this function: if you refactor `z_score` next
month and accidentally break the zero-standard-deviation check, these three tests catch
it immediately, rather than the bug surfacing silently, weeks later, as a wrong number
somewhere downstream. A test suite is, in effect, a safety net that lets you change code
confidently rather than being afraid to touch it.

A related tool, `coverage.py` (`pip install coverage`), reports what fraction of your
actual code was executed by your test suite — a useful way to spot functions or branches
(an `except` clause you never triggered, an `if` condition you never made false) that
have no tests at all. High coverage doesn't guarantee correct tests, but low coverage
reliably flags code nobody has verified.

### Requesting critiques from AI coding tools

Tools like Claude, GitHub Copilot, and similar AI assistants are increasingly part of a
normal code review workflow — not as a replacement for the tools above, but as an
additional reviewer that can read a whole function or file and reason about it in
natural language. They're particularly good at a few specific asks: "what edge cases does
this function not handle?", "is there a simpler way to write this?", and "does this
docstring actually match what the code does?" A few habits make this genuinely useful
rather than just a source of plausible-sounding noise:

- Ask specific questions rather than "is this good?" — "what happens if `values` is
  empty?" gets a more actionable answer than a vague request for general feedback.
- Treat every suggestion as a claim to verify, not an instruction to apply. Run the
  linter and test suite on the result; an AI tool can suggest a change that reads as more
  idiomatic while quietly changing behavior, and the tools earlier in this section are
  exactly what catches that.
- Use it for the things static tools structurally can't check — whether a variable name
  actually communicates its purpose, whether a function is doing too many unrelated
  things at once, whether a docstring has drifted out of sync with the code it
  describes. Linters and type checkers are precise but narrow; an AI reviewer's value is
  breadth and judgment, at the cost of occasionally being confidently wrong — which is
  exactly why the verification step above matters.

### Code review and pair programming

The same principle extends to human collaborators: having someone else read your code
before it's merged catches a different category of problem than any automated tool —
unclear naming, a design that will be painful to extend, an assumption that's true today
but won't be next quarter. A useful norm, whether the reviewer is a person or an AI tool,
is to review for behavior and clarity first (does this do what it claims, and can the
next person understand it?) rather than style, since a formatter already handles style
automatically.

### Defensive programming and assertions

Section 6 covered raising exceptions for genuinely invalid input. A lighter-weight,
related tool is the `assert` statement, typically used to catch programmer errors —
states that should be logically impossible if the rest of the code is correct — rather
than to validate external input.


In [ ]:
def train_test_split_ratio(n_total, n_train):
    assert 0 < n_train < n_total, "n_train must be between 0 and n_total"
    return n_train / n_total

print(train_test_split_ratio(100, 70))

try:
    train_test_split_ratio(100, 150)
except AssertionError as e:
    print(f"assertion failed: {e}")



Unlike a `raise ValueError(...)` aimed at bad external data, an `assert` is meant as an
internal sanity check on your own code's invariants, and is commonly stripped out
entirely in optimized production runs — which is exactly why it should never be used to
validate genuinely untrusted input from a file, an API, or a user, where Section 6's
`try`/`except` and explicit `raise` remain the right tool.

### Keeping functions small and avoiding repetition

Two closely related habits pay off more than almost any specific tool: a function that
does one clearly nameable thing is far easier to test, review, and reuse than one that
does five things at once, and the same handful of lines copy-pasted in three places
(rather than pulled out into one shared function) means three separate places to find and
fix the same bug later — a principle commonly summarized as **DRY: don't repeat
yourself**. Neither habit is enforced by any tool in this section; both are judgment
calls that get easier to make well with practice, and they're usually exactly what a good
code review or AI critique will flag first.

### Version control discipline

Finally, one habit that sits outside any single file: committing your work in small,
focused changes with clear messages makes every tool in this section more useful in
combination. Compare:

```bash
git commit -m "fix off-by-one error in train/test split"   # good: specific, clear intent
git commit -m "updates"                                     # bad: tells you nothing later
```

A failing test or a linter warning is far easier to track down to its cause when it's
introduced by one small, well-described change rather than buried in one enormous commit
touching dozens of files at once.

---

Put together, these tools form layers that catch different kinds of problems: formatters
remove style as a question entirely, linters catch likely mistakes before you run
anything, type checkers catch a class of bugs statically, tests verify actual behavior
and protect you when you change code later, and human or AI review catches the things
none of the automated tools are built to judge. No single layer replaces the others — the
labs ahead will make the most of all of them together.

---

## Where this leaves you

The nine sections of this module cover the complete set of tools you need to read and
write the Python code used throughout this course: values and types, decisions,
repetition, reusable functions, the core collections that hold your data, reliable I/O
and error handling, the package ecosystem this course's stack is built from, the
object-oriented patterns that stack itself is built on top of, and the habits and tools
that keep all of it reliable as it grows. Nothing in the labs ahead will require a
programming concept outside this list — only more sophisticated combinations of exactly
these pieces, applied to larger and more interesting data.
